# Phase 2 — SCRFD detection and official alignment
## Goal
Require exactly one face before computing an ArcFace embedding. Count all SCRFD detections above the confidence gate before quality filtering; never select the largest face from a crowd. No real images or identities are displayed or saved.

## Setup

In [ ]:
from pathlib import Path
import sys, json, os, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/phase2_verification.yaml').exists())
sys.path.insert(0,str(ROOT/'src'))
os.environ['MPLCONFIGDIR'] = str(ROOT/'.cache/matplotlib')
from face_verification import *
config = load_config(root=ROOT)
REPORTS = ROOT/'reports/phase2'
# Real faces are opt-in. Set these manually only after confirming each participant's consent.
RUN_REAL_FACES = False
ADULT_CONSENT = False
IDENTITY_CONSENT = False
PRIVATE_MANIFEST = ROOT/'data/private/consent_manifest.json'
print('Interpreter:',sys.executable)
print('Real-face workflow enabled:',RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT)


PRIVATE_INPUTS = Path(os.environ.get('FACEATTEND_PRIVATE_INPUTS', str(Path.home()/'AppData/Local/FaceAttend/phase2/private')))
# Explicit executor opt-in; ordinary Run All keeps all defaults false.
if os.environ.get('FACEATTEND_CONSENTED_SMOKE') == '1':
    supplied = json.loads((PRIVATE_INPUTS/'input_manifest.json').read_text())
    assert supplied['adult_identity_verification_consent_confirmed_by_user'] is True
    assert supplied['research_terms_confirmed_by_user'] is True
    RUN_REAL_FACES = ADULT_CONSENT = IDENTITY_CONSENT = True
print('Consented three-image smoke enabled:',RUN_REAL_FACES)


## Quality policy
Research defaults: confidence 0.6, input minimum 160 pixels, face box minimum 80 pixels, face-region Laplacian variance ≥60, brightness 35–220, untruncated box and valid five-point landmarks. These are engineering gates, not calibrated biometric thresholds. They need adjustment using development captures only; their changes invalidate the model/policy fingerprint. Severe occlusion and unusual pose can still evade these heuristics. No liveness guarantee.

In [ ]:
print({k:v for k,v in vars(config).items() if k.startswith(('min_','max_','detection_'))})

## Synthetic error-path checks (no face-model inference)

In [ ]:
from types import SimpleNamespace
import numpy as np
session = SimpleNamespace(get_providers=lambda:['CUDAExecutionProvider'],disable_fallback=lambda:None)
image = np.random.default_rng(42).integers(0,255,(200,200,3),dtype=np.uint8)
for count in [0,2]:
    fake = SimpleNamespace(config=config,detector=SimpleNamespace(session=session,
        detect=lambda *args, count=count, **kwargs:(np.zeros((count,5)),None)))
    print('Injected detections:',count,detect_single_face(image,fake).to_dict())
fake = SimpleNamespace(config=config)
print('Unreadable:',detect_single_face(None,fake).to_dict())
print('Low resolution:',detect_single_face(np.zeros((50,50,3),np.uint8),fake).to_dict())

## Consented sample inference — manual opt-in
Set the three consent switches and create the private manifest from the documented schema first. This cell uses the first enrollment capture solely to inspect aggregate quality and alignment dimensions; it emits no portrait, pseudonym, bounding box or landmark coordinates into saved outputs.

In [ ]:
if RUN_REAL_FACES and ADULT_CONSENT and IDENTITY_CONSENT:
    import cv2
    models = initialize_models(config)
    observations = []
    for index,path in enumerate(sorted((PRIVATE_INPUTS/'captures').glob('*_input.jpeg')),1):
        face = detect_single_face(cv2.imread(str(path)),models)
        record = {'input_index':index,**face.to_dict()}
        if face.status == Status.OK:
            record['aligned_dimensions'] = list(face.aligned.shape)
            record['five_landmarks_available'] = face.landmarks.shape == (5,2)
        observations.append(record)
    print(json.dumps(observations,indent=2))
else:
    print('NOT_RUN: use the explicit consented-image executor opt-in for private inference.')

## Checks and next steps
The authorized real diagnostic detects a face in each main photograph. Inputs 1 and 2 fail the predeclared blur gate (variance 7.45 and 20.73 versus minimum 60); input 3 passes (variance 104.72) and aligns to 112x112 using five landmarks. No portraits, coordinates or raw vectors appear in saved output. Screenshots are cropped to their main photograph to exclude all gallery thumbnails; no synthetic capture variants were generated.

Use original clear photos to replace the blurred screenshots. This gate is an engineering heuristic and does not prove absence of occlusion or spoofing.

## Latest ten-photo quality results

This section reads the executed ten-image retest artifact. The initial three-image experiment above is retained as history. User-supplied labels are represented by pseudonyms; no portraits or vectors are displayed. The user confirmed all ten captures are from one session. Enrollment uses photos 1/2/4 and 7/8/9; probes use 3/5 and 6/10, selected before score inspection. The encrypted enrollment smoke used a process-only key and removed all templates afterward.

These results establish implementation behavior, not a calibrated matching decision, independent-session accuracy, liveness, FMR/FNMR reliability or production readiness.

In [ ]:
import pandas as pd
batch = json.loads((REPORTS/'batch02_results.json').read_text())
display(pd.DataFrame([{'photo':r['photo'],'role':r['role'],'status':r['status'],'reason':r['message'],
    'blur_variance':r['quality'].get('blur_variance'),'embedding_dimension':r.get('embedding_dimension')}
    for r in batch['observations']]))
print('Quality passed:',batch['quality_passed'],'of',batch['input_count'])

## Latest single-subject cross-session diagnostic

This section reads `batch03_results.json`, produced by an actual fresh-process GPU run. The user supplied one subject's ten photos and confirmed every photo came from a separate session; exact dates were not supplied. Enrollment photos 1/2/7 and probe photos 3/4/5/6/8/9/10 were fixed before inspecting scores. Photos 6/10 fail the unchanged size/clipping gate. No rejected image is embedded.

The three-image encrypted enrollment, duplicate-ID rejection and deletion passed. Five valid probes have diagnostic genuine scores 0.529–0.835; ten comparisons with explicitly labeled prior controls have scores 0.022–0.268. Session independence is user-confirmed for the new subject; cross-subject chronology is unconfirmed. Comparisons share subjects/templates, so the counts are dependent evidence, not independent population trials. All operational verification attempts refuse with THRESHOLD_NOT_CALIBRATED. No production threshold, FMR/FNMR, liveness or attendance claim is issued.

Names, original images and vectors remain out of public notebook outputs. Test templates and the process-only key were removed. Earlier cohorts above are preserved as history.

In [ ]:
latest = json.loads((REPORTS/'batch03_results.json').read_text())
import pandas as pd
display(pd.DataFrame([{'photo':r['photo'],'role':r['role'],'status':r['status'],'reason':r['message'],'blur_variance':r['quality'].get('blur_variance')} for r in latest['observations']]))
print('Quality passed:',latest['quality_passed'],'of',latest['input_count'])